# Colab Base para el Trabajo Práctico (Entrega 4)

Programa de creación para la entrega 4

In [ ]:
import pandas as pd
import sqlite3

import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import metrics

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# 0. Lectura de datos

In [ ]:
# TODO: Cambiar para que apunte al directorio correcto
DIR = "/content/drive/MyDrive/DM/2026/datos"
# DIR = "/content/drive/MyDrive/datasets/properati"
engine = sqlite3.connect(f"{DIR}/entrenamiento.db")
df_ent = pd.read_sql("SELECT * FROM entrenamiento", engine, index_col='id')

# cantidad de filas y columnas
df_ent.shape

In [ ]:
df_ent.columns

# 1. Entender los datos (AID)

# 2. Limpiar y transformar los datos (DM)

## 2.1. Filtrado de datos



In [ ]:
# Selección de datos. Solo a fines demostrativos.
# TODO: CAMBIAR!
df_ent = df_ent.loc[(df_ent["location_1"] == "Santa Fe") & (df_ent["operation_type"] == 'alquiler')]
df_ent.shape

In [ ]:
# La creación de modelos requiere que no haya valores perdidos
# por eso llenamos todo con 0 a lo bestia
# TODO: Solo se puede modificar por otra constante
df_ent = df_ent.fillna(0)

## 2.2. Tratamiento de valores atípicos

## 2.3. Imputación de valores perdidos

In [ ]:
# TODO: Mejorar la estrategia de imputación.
df_ent = df_ent.fillna(0)

## 2.4. Creación de nuevos atributos

In [ ]:
# Generación de nuevas columnas
# TODO: Crear nuevas variables a partir de datos YA EXISTENTES en el dataset (No usar ningún tipo de fuente externa)
# Tampoco usar librerias ajenas a las que se están viendo en la asignatura.

df_ent["feng-shui"] = 1

## 2.5. Separar Modelos

In [ ]:
# Separar df_ent en dos segmentos por tipo de propiedad
seg_depto = df_ent[df_ent["property_type"] == "departamento"]
seg_casa  = df_ent[df_ent["property_type"] == "casa"]

print(f"departamento: {seg_depto.shape[0]} filas")
print(f"casa:         {seg_casa.shape[0]} filas")

# 3. Entrenamiento del modelos (AA)- ⛔⛔⛔ NO CAMBIAR EL MODELO ⛔⛔⛔

In [ ]:
import matplotlib.pyplot as plt

# Guardar tipo de propiedad antes de quedarnos solo con columnas numéricas
_pt = df_ent["property_type"].copy()
df_ent_num = df_ent.select_dtypes('number')

X = df_ent_num[df_ent_num.columns.drop('price')]
y = df_ent_num['price']

# 3.1. Optimización de hiperparámetros por segmento
mejores_hp = {}
segmentos = ["departamento", "casa"]

# Parametros a probar
param_grid = {
    'n_estimators': [50, 500],
    'max_depth': [5, 25],
    'min_samples_split': [2, 6],
}

fig, axes = plt.subplots(1, len(segmentos), figsize=(15, 6))

for i, nombre in enumerate(segmentos):
    mask = (_pt == nombre)
    X_seg, y_seg = X[mask], y[mask]

    if X_seg.shape[0] == 0:
        print(f"{nombre}: sin datos — omitido")
        continue

    print(f"Optimizando para {nombre} ({X_seg.shape[0]} filas)...")

    # Búsqueda de mejores hiperparámetros
    grid = sk.model_selection.GridSearchCV(
        sk.ensemble.RandomForestRegressor(random_state=42),
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=3,
        n_jobs=-1
    )
    grid.fit(X_seg, y_seg)

    mejores_hp[nombre] = grid.best_params_
    print(f"Mejores HP {nombre}: {grid.best_params_}")

    # 3.2. Análisis de importancia de variables para este segmento
    importancias = pd.Series(grid.best_estimator_.feature_importances_, index=X.columns)
    importancias.nlargest(10).plot(kind='barh', ax=axes[i], title=f'Importancia - {nombre}')

plt.tight_layout()
plt.show()

# 4. Solución para subir Kaggle

In [ ]:
df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")
df_ap.head(2)

In [ ]:
df_ap.shape

In [ ]:
mejores_hp

In [ ]:
# Reentrenar con TODOS los datos (sin split) para la solución final
# Usamos los mejores HP encontrados en el punto 3 para cada tipo de propiedad
models_final = {}
X_cols_final = {}

for nombre, hp in mejores_hp.items():
    mask = (_pt == nombre)
    X_seg, y_seg = X[mask], y[mask]

    if X_seg.shape[0] == 0:
        continue

    reg = sk.ensemble.RandomForestRegressor(**hp, n_jobs=-1, random_state=42)
    reg.fit(X_seg, y_seg)

    models_final[nombre] = reg
    X_cols_final[nombre] = X_seg.columns.tolist()
    print(f"{nombre}: reentrenado con {X_seg.shape[0]} filas (datos completos) usando {hp}")

## 4.2. Generación del archivo para Kaggle

In [ ]:
# Aplicamos en df_ap las mismas transformaciones que en df_ent
df_ap = df_ap.fillna(0)
df_ap["feng-shui"] = 1

# Guardar tipo de propiedad antes de quedarnos solo con columnas numéricas
_pt_ap = df_ap["property_type"].replace({
    'departamentos': 'departamento', 'casas': 'casa', 'cocheras': 'cochera'
})
df_ap = df_ap.select_dtypes('number')

ap_index  = df_ap.index.copy()
y_pred_ap = pd.Series(index=ap_index, dtype=float)

segmentos_pred = {
    "departamento": _pt_ap == "departamento",
    "casa":         _pt_ap == "casa",
}

for nombre, mask in segmentos_pred.items():
    if nombre not in models_final or mask.sum() == 0:
        continue
    X_ap_t = df_ap[mask].reindex(columns=X_cols_final[nombre], fill_value=0)
    y_pred_ap[mask] = models_final[nombre].predict(X_ap_t)
    print(f"{nombre}: {mask.sum()} predicciones")

# Fallback para tipos sin modelo (cochera en nuestro corte por tipo de propiedad)
sin_pred = y_pred_ap.isna()
if sin_pred.sum() > 0:
    print(f"Modelo fallback (departamento) para {sin_pred.sum()} filas")
    X_ap_fb = df_ap[sin_pred].reindex(columns=X_cols_final["departamento"], fill_value=0)
    y_pred_ap[sin_pred] = models_final["departamento"].predict(X_ap_fb)

print(f"\nTotal: {y_pred_ap.notna().sum()} / {len(y_pred_ap)} predicciones")

In [ ]:
# Lleno el precio de df_ap con las predicciones
df_ap["price"] = y_pred_ap

# Grabo el df_ap en un archivo csv para subir a Kaggle
nombre = "base-v4"
df_ap["price"].to_csv(f"{DIR}/solucion-{nombre}.csv")

# 5. Análisis de los errores

In [ ]:
# Analizamos los errores por segmento usando los mejores HP
dfs_error = []

for nombre, hp in mejores_hp.items():
    mask = (_pt == nombre)
    X_seg, y_seg = X[mask], y[mask]

    if X_seg.shape[0] < 5: # Evitar errores con pocos datos
        continue

    X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(
        X_seg, y_seg, test_size=0.2, random_state=42
    )

    reg = sk.ensemble.RandomForestRegressor(**hp, n_jobs=-1, random_state=42)
    reg.fit(X_train, y_train)

    y_pred = reg.predict(X_test)

    # Construir dataframe de error para este segmento
    df_res = X_test.copy()
    df_res["segmento"] = nombre
    df_res["price"] = y_test
    df_res["pred_price"] = y_pred
    df_res["error"] = abs(y_pred - y_test)

    dfs_error.append(df_res)

# Consolidar todos los errores
X_test_final = pd.concat(dfs_error)
X_test_final.sort_values(by="error", ascending=False, inplace=True)

print(f"Error medio global: {X_test_final['error'].mean():.2f}")
X_test_final.head(10)